# A2 | 09–10/10 — Auditar qualidade, datas e elegibilidade

**Pessoa A.** Distinguir problema de qualidade, ausência legítima, variável inadequada e risco de vazamento.

**Saída até 10/10:** relatório de qualidade revisado, flags aprovadas e matriz de elegibilidade v1.0.
**Revisão:** B verifica padrões visuais; C transforma as decisões em validações automáticas. **Gate 1 (09/10):** regras de preparação.

In [1]:
# ----- Setup (não editar) -----
import os, sys
from pathlib import Path

# PKG_ROOT: pasta que contém src/protocolo (onde o pacote foi copiado)
# REPO_ROOT: raiz do repositório (tem .git ou data/); é de onde lemos dados e gravamos saídas
_parents = [Path.cwd(), *Path.cwd().parents]
PKG_ROOT = next((p for p in _parents if (p / "src" / "protocolo").exists()), None)
if PKG_ROOT is None:
    raise FileNotFoundError(
        "Pasta src/protocolo não encontrada acima de " + str(Path.cwd()) + ". "
        "Copie 'src/protocolo/' do pacote para o 'src/' do repositório "
        "(deve existir src/protocolo/verificacoes.py) e rode a célula de novo.")
REPO_ROOT = next((p for p in _parents if (p / ".git").exists() or (p / "data").exists()), PKG_ROOT)
os.chdir(REPO_ROOT); sys.path.insert(0, str(PKG_ROOT))

# Proteção contra outro pacote chamado 'src' (ex.: instalado por pip) ou 'src' antigo em cache no kernel:
# um pacote regular em qualquer ponto do sys.path vence um 'src' sem __init__.py.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
_init = PKG_ROOT / "src" / "__init__.py"
if not _init.exists():
    _init.touch()
    print("Criado", _init, "(torna o src do projeto um pacote regular)")

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from src.protocolo import verificacoes as v
if getattr(v, "MODULE_VERSION", "0") < "0.8":
    raise RuntimeError(
        "src/protocolo/verificacoes.py está desatualizado (versão " + str(getattr(v, "MODULE_VERSION", "antiga")) + "). "
        "Substitua pelo arquivo do pacote novo e REINICIE o kernel (o Python mantém o módulo antigo em cache).")

# ----- Parâmetros -----
USAR_DEMO = False                       # True = dados sintéticos só para testar o notebook
DATA_PATH = None                        # None = procura Parquet/CSV em data/ ; ou informe o caminho do arquivo
SAMPLE = 0                              # ex.: 200_000 para execução rápida
OUT = "outputs/tables/pessoa_a"         # saídas (tabelas, atas, decisões), relativas à raiz do repositório
CFG_PATH = next((str(p) for p in [REPO_ROOT / "configs" / "decisoes.yaml", PKG_ROOT / "configs" / "decisoes.yaml"]
                 if p.exists()), "configs/decisoes.yaml")
SUPERVISED_CONFIG = "configs/supervised.yaml"   # config do C (alvo, features, coluna_tempo)
COLUNAS = "classificadas"                   # "minimas" (menos memória) | "classificadas" | "todas" (pesado!) | lista de nomes
SUPERVISED_CONFIG = "configs/supervised.yaml"
EXCLUIR_PROIBIDAS = True        # v1.0: nenhuma coluna proibida segue para o C
print("Raiz do repositório:", REPO_ROOT)
print("Pacote (src/protocolo):", PKG_ROOT)
print("Config de decisões:", CFG_PATH)
cfg = v.load_config(CFG_PATH)
rep = v.Report(OUT, "relatorio_A2")
df = v.prepare(v.carregar_base(DATA_PATH, USAR_DEMO, SAMPLE, COLUNAS, cfg, SUPERVISED_CONFIG))
print(f"Memória do DataFrame: {v.uso_memoria_gb(df):.2f} GB")

Raiz do repositório: c:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
Pacote (src/protocolo): c:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
Config de decisões: c:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\configs\decisoes.yaml
[base] C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\processed\dados-v2\srag.parquet -> 4,666,910 linhas x 110 colunas carregadas (de 198 no arquivo) | leitura compacto (texto em category)
Memória do DataFrame: 1.03 GB


## 1. Vazios, ignorados e campos inexistentes por ano

In [2]:
v.a2_missing(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a2_campos_inexistentes_por_ano.csv").head(40))

[!!] A2 | Campo inexistente por ano: 87 campos 100% vazios em algum ano (ex.: ANTIVIRAL, ASMA, AVE_SUINO, CARDIOPATI, CLASSI_FIN)


,coluna,1917,1918,1941,2009,2011,2012,2016,2017,2018,...,2032,2033,2034,2050,2052,2055,2056,2092,2098,2099
0,ANTIVIRAL,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.00,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ASMA,100.0,100.0,50.0,0.0,100.0,50.0,0.0,0.0,90.91,...,100.0,66.67,100.0,0.0,100.0,100.0,100.0,100.0,100.0,0.0
2,AVE_SUINO,50.0,100.0,25.0,0.0,0.0,0.0,0.0,0.0,27.27,...,0.0,66.67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,CARDIOPATI,100.0,100.0,50.0,0.0,100.0,50.0,0.0,0.0,90.91,...,100.0,33.33,0.0,0.0,100.0,100.0,100.0,100.0,100.0,0.0
4,CLASSI_FIN,25.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,9.09,...,0.0,33.33,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
5,CLASSI_OUT,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.00,...,100.0,100.00,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
6,CO_REGIONA,25.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,9.09,...,0.0,33.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,CO_RG_INTE,25.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,9.09,...,0.0,66.67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,CO_RG_RESI,25.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,9.09,...,0.0,33.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,CRITERIO,50.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,9.09,...,0.0,33.33,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Coerência de datas e idade
Classifique cada inconsistência: **excluir**, **transformar em ausente**, **manter com flag** ou **apenas documentar** (a coluna `acao_sugerida` é uma proposta).

In [3]:
v.a2_dates(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a2_coerencia_datas.csv").head(40))

[!!] A2 | Coerência de datas: 2 regra(s) com violações relevantes: DT_NASC <= DT_SIN_PRI (1); DT_EVOLUCA <= DT_ENCERRA (107196)


,regra,descricao,n_comparaveis,n_violacoes,pct,acao_sugerida
0,DT_NASC <= DT_SIN_PRI,Nascimento posterior ao início dos sintomas,4662951,1,0.000,excluir
1,DT_SIN_PRI <= DT_NOTIFIC,Início dos sintomas posterior à notificação,4666910,0,0.000,manter com flag
2,DT_SIN_PRI <= DT_INTERNA,Início dos sintomas posterior à internação,4362603,24340,0.558,manter com flag
3,DT_INTERNA <= DT_EVOLUCA,Evolução anterior à internação,3822766,0,0.000,excluir
4,DT_EVOLUCA <= DT_ENCERRA,Encerramento anterior à evolução,3985787,107196,2.689,manter com flag
5,DT_NASC válida,"Data futura, não interpretável ou fora de 1900...",4662964,13,0.000,transformar em ausente
6,DT_SIN_PRI válida,"Data futura, não interpretável ou fora de 1900...",4666910,0,0.000,transformar em ausente
7,DT_NOTIFIC válida,"Data futura, não interpretável ou fora de 1900...",4666910,0,0.000,transformar em ausente
8,DT_INTERNA válida,"Data futura, não interpretável ou fora de 1900...",4362665,108,0.002,transformar em ausente
9,DT_EVOLUCA válida,"Data futura, não interpretável ou fora de 1900...",4014740,0,0.000,transformar em ausente


## 3. Duplicidades
`NU_NOTIFIC` repetido **não** implica mesma pessoa.

In [4]:
v.a2_duplicates(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a2_duplicidades.csv").head(40))

[ok] A2 | Duplicidades: 0 linhas exatas duplicadas; demais tipos em a2_duplicidades.csv


,tipo,n_linhas,n_grupos
0,linhas exatamente duplicadas,0,0
1,NU_NOTIFIC repetido (não implica mesma pessoa),58,29
2,"mesma chave ['NU_NOTIFIC', 'DT_NOTIFIC', 'DT_N...",50,25


## 4. Maturidade de EVOLUCAO por mês de internação
Meses recentes do holdout podem ter evolução ainda aberta.

In [5]:
v.a2_maturity(df, cfg, rep)
display(pd.read_csv(Path(OUT) / 'a2_maturidade_evolucao_por_mes.csv').tail(24))

[ok] A2 | Maturidade de EVOLUCAO (holdout): todos os meses do holdout com evolução madura


,mes,n,pct_evolucao_fechada,pct_com_dt_encerra,mediana_dias_ate_encerra
116,2028-04,1,100.0,0.0,NaN
117,2028-06,1,100.0,100.0,-1088.0
118,2028-09,1,100.0,100.0,-1093.0
119,2028-11,1,100.0,100.0,-1086.0
120,2029-03,1,100.0,100.0,-3253.0
121,2029-05,1,0.0,100.0,-1094.0
122,2029-06,1,100.0,100.0,-1074.0
123,2029-07,2,0.0,0.0,NaN
124,2029-09,2,0.0,0.0,NaN
125,2029-10,1,0.0,100.0,-1441.0


## 5. Matriz de elegibilidade
Classes: **incluir**, **incluir com condição**, **auditoria**, **excluir**. Para cada variável responda: *quando foi observada, quando chegou à base e se poderia existir na admissão.* Preencha as colunas em branco do CSV gerado (`a2_matriz_elegibilidade.csv`).

In [6]:
v.a2_eligibility(df, cfg, rep)
display(pd.read_csv(Path(OUT) / "a2_matriz_elegibilidade.csv").head(40))

[!!] A2 | Matriz de elegibilidade: {'auditoria': 93, 'incluir com condição': 39, 'incluir': 34, 'excluir': 32} — preencher colunas em branco e fechar 'auditoria'


,coluna,classe,justificativa,existe_na_admissao_sugerido,pct_vazio,n_distintos_amostra,quando_observada,quando_chegou_na_base,decisao_final,responsavel,data
0,AMOSTRA,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ANO_ARQUIVO,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AN_ADENO,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AN_OUTRO,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AN_PARA1,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AN_PARA2,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,AN_PARA3,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,AN_SARS2,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,AN_VSR,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ARQUIVO_ORIGEM,auditoria,"Existe no Parquet, mas não foi carregada (fora...",Verificar,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Tabela de decisões

In [7]:
E = "A2"
v.log_decision(OUT, "D-06", "Ação por tipo de inconsistência de datas",
               "; ".join(f"{r[2]} -> {r[3]}" for r in cfg["date_rules"]), E,
               impacto="Ver a2_coerencia_datas.csv")
v.log_decision(OUT, "D-07", "Duplicidades", "Linhas exatas removidas; NU_NOTIFIC repetido apenas documentado", E,
               impacto="Ver a2_duplicidades.csv")
v.log_decision(OUT, "D-08", "Maturidade de EVOLUCAO",
               f"Meses do holdout com < {cfg['maturity_min_closed_pct']}% de evolução fechada ficam fora ou sob sensibilidade", E,
               impacto="Ver a2_maturidade_evolucao_por_mes.csv")
v.log_decision(OUT, "D-09", "Variáveis proibidas/condicionais",
               "Matriz de elegibilidade v1.0 (classes incluir/condição/auditoria/excluir)", E,
               impacto="C bloqueia automaticamente (test_leakage); B usa na matriz de clustering")
display(pd.read_csv(Path(OUT) / "tabela_decisoes.csv"))

,id,etapa,tema,decisao,responsavel,data,status,impacto,evidencia
0,D-01,A1,Ponto de previsão,DT_INTERNA (admissão); campos posteriores bloq...,Pessoa A,2026-10-16,proposta,Define quais variáveis podem entrar,Dicionário SIVEP-Gripe (completar)
1,D-02,A1,Coorte estrita,"HOSPITAL=1, DT_INTERNA válida, período analíti...",Pessoa A,2026-10-16,proposta,Ver waterfall (a1_waterfall.csv),NaN
2,D-03,A1,Alvo principal,Óbito SRAG (EVOLUCAO=2) vs cura (EVOLUCAO=1),Pessoa A,2026-10-16,proposta,Alinhar target.positivos do supervised.yaml (G1),NaN
3,D-04,A1,Evolução desconhecida e óbito outras causas,EVOLUCAO=9/ausente e 3 saem do alvo binário; n...,Pessoa A,2026-10-16,proposta,"Exigir filtro EVOLUCAO in {1,2} no loader do C",NaN
4,D-12,A1,Features derivadas (criadas no dataset analítico),"idade_anos (TP_IDADE), faixa_etaria, intervalo...",Pessoa A,2026-10-16,proposta,Só informação da admissão; o C2 as encontra no...,NaN
5,D-05,A1,Período analítico,"Desenvolvimento [2022, 2023, 2024]; holdout [2...",Pessoa A,2026-10-16,proposta,Exclui anos fora do período; alinhar G2,NaN
6,D-06,A2,Ação por tipo de inconsistência de datas,Nascimento posterior ao início dos sintomas ->...,Pessoa A,2026-10-16,proposta,Ver a2_coerencia_datas.csv,NaN
7,D-07,A2,Duplicidades,Linhas exatas removidas; NU_NOTIFIC repetido a...,Pessoa A,2026-10-16,proposta,Ver a2_duplicidades.csv,NaN
8,D-08,A2,Maturidade de EVOLUCAO,Meses do holdout com < 90.0% de evolução fecha...,Pessoa A,2026-10-16,proposta,Ver a2_maturidade_evolucao_por_mes.csv,NaN
9,D-09,A2,Variáveis proibidas/condicionais,Matriz de elegibilidade v1.0 (classes incluir/...,Pessoa A,2026-10-16,proposta,C bloqueia automaticamente (test_leakage); B u...,NaN


## 7. Exportar a população analítica (v1.0)
Com a matriz de elegibilidade fechada, regrave o dataset do C. Com `EXCLUIR_PROIBIDAS = True`, features classificadas como **excluir** não seguem para o C (o dataset lista o que ficou de fora).

In [8]:
info = v.a1_export_dataset(df, cfg, rep, SUPERVISED_CONFIG, versao="1.0", excluir_proibidas=EXCLUIR_PROIBIDAS)
print("Arquivo:", info["arquivo"], "|", f"{info['n_linhas']:,} linhas")
print("Features ausentes na base:", info["features_ausentes_na_base"] or "nenhuma")
print("Colunas:", info["colunas"])
v.validar_contrato_c2(info["arquivo"], SUPERVISED_CONFIG, rep)   # contrato do C2 (colunas, datas, alvo, leakage)

[ok] A1 | Features derivadas: criadas no dataset: idade_anos, faixa_etaria
[!!] A1 | Dataset analítico: 1,236,075 linhas v1.0; features ausentes na base: DIAS_SINTOMAS_ATE_INTERNACAO, SEXO_CATEGORIA
Arquivo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\analytical\dataset_supervisionado.parquet | 1,236,075 linhas
Features ausentes na base: ['DIAS_SINTOMAS_ATE_INTERNACAO', 'SEXO_CATEGORIA']
Colunas: ['nu_notific', 'dt_interna', 'evolucao', 'idade_anos', 'cs_gestant', 'cs_raca', 'cs_zona', 'ave_suino', 'febre', 'tosse', 'garganta', 'dispneia', 'diarreia', 'vomito', 'dor_abd', 'fadiga', 'perd_olft', 'perd_pala', 'outro_sin', 'puerpera', 'cardiopati', 'hematologi', 'sind_down', 'hepatica', 'asma', 'diabetes', 'neurologic', 'pneumopati', 'imunodepre', 'renal', 'obesidade', 'out_morbi', 'sg_uf', 'faixa_etaria', 'ano_internacao']
[XX] A1-C2 | Colunas exigidas pelo C2: ausentes no dataset: dias_sintomas_ate_internacao, sexo_categoria
[ok] A1-C2 | Coluna de tempo:

False

## 8. Fechamento

In [9]:
path = rep.write_md()
print("Relatório:", path)
display(pd.DataFrame(rep.findings, columns=["etapa", "verificação", "status", "detalhe"]))

Relatório: outputs\tables\pessoa_a\relatorio_A2.md


,etapa,verificação,status,detalhe
0,A2,Campo inexistente por ano,ATENÇÃO,87 campos 100% vazios em algum ano (ex.: ANTIV...
1,A2,Coerência de datas,ATENÇÃO,2 regra(s) com violações relevantes: DT_NASC <...
2,A2,Duplicidades,OK,0 linhas exatas duplicadas; demais tipos em a2...
3,A2,Maturidade de EVOLUCAO (holdout),OK,todos os meses do holdout com evolução madura
4,A2,Matriz de elegibilidade,ATENÇÃO,"{'auditoria': 93, 'incluir com condição': 39, ..."
5,A1,Features derivadas,OK,"criadas no dataset: idade_anos, faixa_etaria"
6,A1,Dataset analítico,ATENÇÃO,"1,236,075 linhas v1.0; features ausentes na ba..."
7,A1-C2,Colunas exigidas pelo C2,FALHA,ausentes no dataset: dias_sintomas_ate_interna...
8,A1-C2,Coluna de tempo,OK,"datetime válido, ordenado: 2022-01-01 -> 2025-..."
9,A1-C2,Alvo,OK,apenas cura/óbito; positivos [2.0] presentes


### Checklist de saída (10/10)
- [ ] Relatório de qualidade revisado e flags aprovadas
- [ ] Matriz de elegibilidade v1.0 preenchida (nenhuma variável em `auditoria`)
- [ ] Lista de variáveis proibidas e condições de saturação, vacinação e geografia fechadas
- [ ] Dataset analítico v1.0 regravado e C avisado
- [ ] Regras entregues a B e C